# 03. Model Evaluation & Error Analysis

**Th?nh vi?n ph? tr?ch:** Th?nh vi?n 3 - Model Evaluation & Performance Analysis

**M?c ti?u:**
1. N?p file d? ?o?n `reports/predictions.csv` (`load_predictions`).
2. T?nh to?n c?c ch? s? ph?n lo?i t?ng th?: Precision, Recall, F1, PR-AUC, ROC-AUC (`compute_classification_metrics`, `evaluate_all_models`).
3. ??nh gi? ?? ch?nh x?c Top-K theo t?ng m?a gi?i (`compute_topk_accuracy_per_season`) m? ph?ng 24 su?t All-Star th?c t?.
4. Ph?n t?ch c?c tr??ng h?p sai l?ch (Error Cases) (`get_error_cases`):
   - False Positives (X?c su?t cao nh?ng kh?ng ???c ch?n).
   - False Negatives (C?u th? All-Star th?c t? b? m? h?nh b? s?t).
5. Tr?c quan h?a so s?nh v? l?u bi?u ?? v?o `figures/evaluation/`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluate import (
    load_predictions,
    compute_classification_metrics,
    compute_topk_accuracy_per_season,
    get_error_cases,
    evaluate_all_models
)

print("Imports th?nh c?ng t? src.evaluate!")

## B??c 1: N?p file d? ?o?n `reports/predictions.csv` (`load_predictions`)

In [ ]:
predictions_path = ROOT_DIR / "reports" / "predictions.csv"
pred_df = load_predictions(predictions_path)

print(f"?? n?p {len(pred_df)} d?ng d? ?o?n.")
print(f"C?c m? h?nh: {pred_df['model'].unique().tolist()}")
print(f"C?c splits:  {pred_df['split'].unique().tolist()}")
pred_df.head()

## B??c 2: B?ng x?p h?ng hi?u n?ng m? h?nh (`evaluate_all_models`)
??nh gi? tr?n t?p Validation v?i ng??ng c?t x?c su?t 0.5:

In [ ]:
leaderboard_val = evaluate_all_models(pred_df, threshold=0.5, split="val")
print("B?ng x?p h?ng hi?u n?ng tr?n t?p Validation:")
leaderboard_val

## B??c 3: ??nh gi? ?? ch?nh x?c Top-K theo t?ng m?a gi?i (`compute_topk_accuracy_per_season`)
V?i m?i m?a gi?i, l?y $k = \sum y_{true}$ (s? All-Star th?c t? c?a m?a ??, th?ng th??ng 24-28 c?u th? t?nh c? ch?n th??ng), ch?n $k$ c?u th? c? x?c su?t cao nh?t v? t?nh t? l? tr?ng kh?p.

In [ ]:
topk_df = compute_topk_accuracy_per_season(pred_df[pred_df["split"] == "val"])
print("?? ch?nh x?c Top-K t?ng m?a gi?i:")
topk_df.head(12)

## B??c 4: Ph?n t?ch c?c tr??ng h?p l?i (Error Cases: False Positives & False Negatives) (`get_error_cases`)
- **False Positives (FP):** X?c su?t cao nh?ng kh?ng ???c ch?n (c?u th? overhyped ho?c b? phi?u kh?ng b?u).
- **False Negatives (FN):** ???c ch?n All-Star nh?ng m? h?nh cho x?c su?t th?p (snubs/underrated).

In [ ]:
# Ph?n t?ch l?i cho m? h?nh c? mean_topk_acc cao nh?t
best_model_name = leaderboard_val.iloc[0]["model"]
target_preds = pred_df[(pred_df["model"] == best_model_name) & (pred_df["split"] == "val")]

errors = get_error_cases(target_preds, threshold=0.5)
fp_df = errors["false_positives"]
fn_df = errors["false_negatives"]

print(f"=== PH?N T?CH L?I CHO M? H?NH: {best_model_name} ===")
print(f"S? l??ng False Positives (D? ?o?n All-Star nh?ng kh?ng ph?i): {len(fp_df)}")
print(f"S? l??ng False Negatives (All-Star th?c t? b? b? s?t):        {len(fn_df)}")

print("\nTop 5 False Positives:")
print(fp_df[["player", "season", "y_true", "y_proba"]].head(5))

print("\nTop 5 False Negatives:")
print(fn_df[["player", "season", "y_true", "y_proba"]].head(5))

## B??c 5: Tr?c quan h?a v? l?u bi?u ?? v?o `figures/evaluation/`

In [ ]:
figures_eval_dir = ROOT_DIR / "figures" / "evaluation"
figures_eval_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=leaderboard_val, x="model", y="mean_topk_acc", palette="viridis")
plt.title("So s?nh Mean Top-K Accuracy gi?a c?c m? h?nh (Validation Split)", fontsize=13)
plt.xlabel("M? h?nh", fontsize=11)
plt.ylabel("Mean Top-K Accuracy", fontsize=11)
plt.ylim(0, 1.05)
for i, row in leaderboard_val.iterrows():
    acc = row["mean_topk_acc"]
    if not np.isnan(acc):
        plt.text(i, acc + 0.02, f"{acc:.2%}", ha="center", fontweight="bold")
plt.tight_layout()

chart_path = figures_eval_dir / "models_topk_accuracy.png"
plt.savefig(chart_path, dpi=300)
print(f"?? l?u bi?u ??: {chart_path}")
plt.show()